# Malaria Classification Task (Infected or UnInfected)

<p> Colored Picture usually have 3 channels - Red, Blue and Green. Each pixel is made of a combination of these 3 channels. 
So our tensor for a colored picture will be (H,W,3): 3 is for 3 channels. A gray scale image has only 1 channel, therefore we have (H,W,1)</p>


<p>(Black = 0) --- (White = 255)</p>
<p>After Normalization, (Black = 0) --- (White = 1)</p>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf # type: ignore
import tensorflow_datasets as tfds # type: ignore

# MODELS
Model = tf.keras.Model

#LAYERS
Input = tf.keras.layers.Input
Normalization = tf.keras.layers.Normalization

Dense = tf.keras.layers.Dense
InputLayer = tf.keras.layers.InputLayer
Conv2D = tf.keras.layers.Conv2D
MaxPool2D = tf.keras.layers.MaxPool2D
Flatten = tf.keras.layers.Flatten # Flatten is a function so you don't need a parantheses at the end.
BatchNormalization = tf.keras.layers.BatchNormalization
Layer = tf.keras.layers.Layer


#LOSSES
bce = tf.keras.losses.BinaryCrossentropy() # To instantiate a class you need to build an object which can only be done by including a parantheses at the end. 

# OPTIMIZER
Adam = tf.keras.optimizers.Adam

In [ ]:
dataset, dataset_info = tfds.load('malaria', with_info=True, as_supervised = True, shuffle_files=True, split=['train']) # tfds_load returns the dataset and dataset_info | 'with_info=True' gives the dataset_info
# as_supervised -> if True returns a dataset with a 2 tuple structure (input, label)

In [ ]:
dataset # this is a list with 1 element, so only select dataset[0]

In [ ]:
dataset_info

In [ ]:
for data in dataset[0].take(2):
    print(data)

### Creating the Train, Test, Validation Datasets

In [ ]:
def splits(dataset, TRAIN_RATIO, VAL_RATIO, TEST_RATIO):
    DATASET_SIZE = len(dataset)

    train_dataset =  dataset.take(int(TRAIN_RATIO * DATASET_SIZE)) # TRAINING DATASET

    val_test_dataset = dataset.skip(int(TRAIN_RATIO * DATASET_SIZE))

    val_dataset = val_test_dataset.take(int(VAL_RATIO*DATASET_SIZE)) # VALIDATION DATASET
    test_dataset = val_test_dataset.skip(int(TEST_RATIO * DATASET_SIZE)) # TEST DATASET

    return train_dataset, val_dataset, test_dataset



In [ ]:
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
TEST_RATIO = 0.2

# dataset = tf.data.Dataset.range(10)

train_dataset, val_dataset, test_dataset = splits(dataset[0], TRAIN_RATIO, VAL_RATIO, TEST_RATIO)


print('Train Dataset\n', list(train_dataset.take(1).as_numpy_iterator()),'Val Dataset\n', list(val_dataset.take(1).as_numpy_iterator()),'Test Dataset\n', list(test_dataset.take(1).as_numpy_iterator()))

In [ ]:
for image, label in train_dataset.take(16):
    print("Image Matrix Shape:", image.shape) 
    print("Label Value:", label.numpy())

# Data Visualization

In [ ]:
for i, (image, label) in enumerate(train_dataset.take(16)): # when using .take(), it goes from 1 - 16, hence the 'i+1'
    ax = plt.subplot(4,4, i+1) #(4,4) because we have 16 images
    plt.imshow(image)
    plt.title(dataset_info.features['label'].int2str(label))
    plt.axis('off')

In [ ]:
dataset_info.features['label'].int2str(0) # our label of 0 means that it is parasitized

In [ ]:
dataset_info.features['label'].int2str(1) # our label of 1 means that it is uninfected

# Data Pre - Processing

##### Image Resizing -> Converting each image to 224,224 resolution

##### Standardization or Normalization

In [ ]:
IM_SIZE = 224
def resize_rescale(image, label):
    return tf.image.resize(image,(IM_SIZE, IM_SIZE))/255.0, label
# We are dividing by 255 to rescale

In [ ]:
train_dataset = train_dataset.map(resize_rescale)
val_dataset = val_dataset.map(resize_rescale)
test_dataset = test_dataset.map(resize_rescale)


In [ ]:
for image, label in train_dataset.take(1):
    print(image, label)
# Image is resized

## Shuffling the dataset

In [ ]:
train_dataset = train_dataset.shuffle(buffer_size=8, reshuffle_each_iteration=True).batch(32).prefetch(tf.data.AUTOTUNE)

# Don't reshuffle val_dataset, so no 'reshuffle_each_iteration=True'
val_dataset = val_dataset.shuffle(buffer_size=8).batch(32).prefetch(tf.data.AUTOTUNE)

# Don't do anything with the test_dataset

In [ ]:
print(train_dataset,'\n', val_dataset)
print(f'Length of training Dataset: {len(train_dataset)}')
print(f'Length of Validation Dataset: {len(val_dataset)}')

# Model Creation

### Sequential API

In [ ]:
# LeNet Architecture
# The Models weights, are the filters that we learn
lenet_model = tf.keras.Sequential([
                            InputLayer(shape = (IM_SIZE, IM_SIZE,3)), 
                                   
                            Conv2D(filters = 6, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Conv2D(filters = 16, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Flatten(), 

                            Dense(100, activation='relu'),
                            BatchNormalization(),

                            Dense(10, activation='relu'),
                            BatchNormalization(),

                            Dense(1, activation='sigmoid')
                            ]) 

### Functional API

In [ ]:
# PUTTING ALL THE FEATURE EXTRACTOR LAYER INTO 'feature_extractor_model'
func_input = Input(shape = (IM_SIZE, IM_SIZE,3), name = 'Input Image')

x = Conv2D(filters = 6, kernel_size = 3, strides= 1, padding='valid', activation='relu', name = '1st Convolution Layer')(func_input)
x = BatchNormalization(name = '1st Normalization Layer')(x) 
x = MaxPool2D(pool_size= 2, strides=2, name = '1st Pooling Layer')(x)
x = Conv2D(filters = 16, kernel_size = 3, strides= 1, padding='valid', activation='relu', name = '2nd Convolution Layer')(x)
x = BatchNormalization(name = '2nd Normalization Layer')(x)

output = MaxPool2D(pool_size= 2, strides=2, name = 'output')(x)

feature_extractor_model = Model(func_input, output, name = 'Feature_Extractor')
feature_extractor_model.summary()

## Adding the Feature Extractor Model to the Classification Layers

In [ ]:
func_input = Input(shape = (IM_SIZE, IM_SIZE,3), name = 'Input Image')

x = feature_extractor_model(func_input)


# Classification Layers
x = Flatten()(x) 

x = Dense(100, activation='relu')(x)
x = BatchNormalization()(x)

x = Dense(10, activation='relu')(x)
x = BatchNormalization()(x)

func_output = Dense(1, activation='sigmoid')(x)


# Need to mention that 'lenet_model_func' is a MODEL with 'func_input' and 'func_output'
lenet_model_func = Model(func_input, func_output, name = 'Lenet_Model')
lenet_model_func.summary()


# Featur Extractor - Sequential Model

In [ ]:
feature_extractor_seq_model = tf.keras.Sequential([
                            InputLayer(shape = (IM_SIZE, IM_SIZE,3)), 
                                   
                            Conv2D(filters = 6, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Conv2D(filters = 16, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2)])

x = feature_extractor_seq_model(func_input)


# Classification Layers
x = Flatten()(x) 

x = Dense(100, activation='relu')(x)
x = BatchNormalization()(x)

x = Dense(10, activation='relu')(x)
x = BatchNormalization()(x)

func_output = Dense(1, activation='sigmoid')(x)

lenet_model_func = Model(func_input, func_output, name = 'Lenet_Model')
lenet_model_func.summary()

In [ ]:
# tf.keras.utils.plot_model(lenet_model, to_file='model.png', show_shapes=True)

In [ ]:
'''lenet_model = tf.keras.Sequential([
                            InputLayer(shape = (IM_SIZE, IM_SIZE,3)), 
                                   
                            Conv2D(filters = 6, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Conv2D(filters = 16, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Flatten(), 

                            Dense(100, activation='relu'),
                            BatchNormalization(),

                            Dense(10, activation='relu'),
                            BatchNormalization(),

                            Dense(1, activation='sigmoid')
                            ]) '''
lenet_model.summary()

# Model SubClassing

In [ ]:
class FeatureExtractor(Layer): # FeatureExtractor inherits from Layer
    def __init__(self, filters, kernel_size, strides, padding, activation, pool_size):
        super(FeatureExtractor, self).__init__() # This line reaches up to the parent Keras Layer class and boots up its internal initialization configuration.
        '''A native Keras layer has a ton of invisible, complex machinery required to track weight gradients and shape states.
           If you omit this line, your custom layer's setup will overwrite the core Keras setup, completely breaking the layer. Running super().__init__()
           ensures that Keras sets up its vital internal plumbing before you start adding your custom layers underneath it.'''
        


        # All the layers are being DEFINED here
        
        self.conv_1 = Conv2D(filters = filters, kernel_size = kernel_size, strides = strides, padding = padding, activation = activation)
        # we are equating filters = filters because we don't want to mix up the argument values here and how it is defined in the documentation
        self.batch_1 = BatchNormalization()
        self.pool_1 = MaxPool2D(pool_size = pool_size, strides = 2 * strides)

        self.conv_2 = Conv2D(filters = filters * 2, kernel_size = kernel_size, strides = strides, padding = padding, activation = activation)
        self.batch_2 = BatchNormalization()
        self.pool_2 = MaxPool2D(pool_size = pool_size, strides = 2 * strides)


    def call(self, x, training = False): 
        x = self.conv_1(x)
        x = self.batch_1(x)
        x = self.pool_1(x)

        x = self.conv_2(x)
        x = self.batch_2(x)
        x = self.pool_2(x)

        return x
        

feature_sub_classed = FeatureExtractor(8, 3, 1, 'valid', 'relu', 2)

In [ ]:
func_input = Input(shape = (IM_SIZE, IM_SIZE,3), name = 'Input Image')

x = feature_sub_classed(func_input) 
# When you pass func_input into feature_sub_classed(...), 
# Keras automatically jumps inside your custom class and runs the 'call' method under the hood.

x = Flatten()(x) 

x = Dense(100, activation='relu')(x)
x = BatchNormalization()(x)

x = Dense(10, activation='relu')(x)
x = BatchNormalization()(x)

func_output = Dense(1, activation='sigmoid')(x)

lenet_model_func = Model(func_input, func_output, name = 'Lenet_Model: Feature Subclassing')
lenet_model_func.summary()

In [ ]:
# Adding the Classification layer to make the whole LeNet Model

class LenetModel(Model):
  def __init__(self):
    super(LenetModel, self).__init__()

    self.feature_extractor = FeatureExtractor(8, 3, 1, "valid", "relu", 2)

    self.flatten = Flatten()

    self.dense_1 = Dense(100, activation = "relu")
    self.batch_1 = BatchNormalization()

    self.dense_2 = Dense(10, activation = "relu")
    self.batch_2 = BatchNormalization()

    self.dense_3 = Dense(1, activation = "sigmoid") # parameter 1: number of output units
    
  def call(self, x, training = False):

    x = self.feature_extractor(x)
    x = self.flatten(x)
    x = self.dense_1(x)
    x = self.batch_1(x)
    x = self.dense_2(x)
    x = self.batch_2(x)
    x = self.dense_3(x)

    return x





# Final LeNet Model using Feature Extractor + Classification Layer 
lenet_sub_classed = LenetModel()
lenet_sub_classed(tf.zeros([1,224,224,3]))
lenet_sub_classed.summary()

# Custom Layers

In [ ]:
# Instead of relying on tf.keras.layers.Dense, we are manually implementing the foundational linear algebra equation of deep learning
class NeuralearnDense(Layer): # Making the custom DENSE Layer
    def __init__(self, output_units, activation):
        super(NeuralearnDense, self).__init__()
        self.output_units = output_units # gives the number of columns
        self.activation = activation
    
    def build(self, input_features_shape): #Important '9:19:05'                                     INPUT: (Batch Size, Features), W: (Features,1), Bias: (Batch Size,1)
        #                                                                                           So Y = INPUT * W + B for the Dense Layer
                                        #      number of rows        number of columns
                                        #            |                      |
                                        #            v                      v
        self.w = self.add_weight(shape = (input_features_shape[-1], self.output_units), initializer='random_normal', trainable=True) # NEED TO GET THE SHAPES RIGHT 
        # random normal weight initializer
        self.b = self.add_weight(shape = (self.output_units, ), initializer='random_normal', trainable=True)

    def call(self, input_features): # This handles the actual forward pass math whenever data flows through the layer.

        pre_output = tf.matmul(input_features, self.w) + self.b # IMPORTANT: THE FORMULA FOR THE DENSE LAYER

        if (self.activation == 'relu'):
            return tf.nn.relu(pre_output)
        elif (self.activation == 'sigmoid'):
            return tf.math.sigmoid(pre_output)
        else:
            pre_output

In [ ]:
# Using our custom DENSE LAYER
lenet_model_custom_dense = tf.keras.Sequential([
                            InputLayer(shape = (IM_SIZE, IM_SIZE,3)), 
                                   
                            Conv2D(filters = 6, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Conv2D(filters = 16, kernel_size = 3, strides= 1, padding='valid', activation='relu'),
                            BatchNormalization(), 
                            MaxPool2D(pool_size= 2, strides=2),

                            Flatten(), 

                            NeuralearnDense(100, activation='relu'),
                            BatchNormalization(),

                            NeuralearnDense(10, activation='relu'),
                            BatchNormalization(),

                            NeuralearnDense(1, activation='sigmoid')
                            ]) 

lenet_model_custom_dense.summary()

# Error Sanctioning -> Binary Cross Entropy Loss

In [ ]:
y_true = [0,1,0,0] # hello ok

y_pred = [0.6, 0.51, 0.94, 1]

bce(y_true, y_pred) # use from_logits = True when the predictions are above 1 and lesser than 0.

### Metrics - Precision, Recall and Accuracy

In [ ]:
lenet_model.compile(optimizer= Adam(learning_rate = 0.01), loss = 'bce', metrics = ['accuracy']) #metrics = [RootMeanSquaredError()]

In [ ]:
lenet_model_func.compile(optimizer= Adam(learning_rate = 0.01), loss = 'bce', metrics = ['accuracy']) #metrics = [RootMeanSquaredError()]

In [ ]:
History = lenet_model.fit(train_dataset, validation_data=val_dataset, epochs=10, verbose=1)

In [ ]:
History2 = lenet_model_func.fit(train_dataset, validation_data=val_dataset, epochs=10, verbose=1) # Training the model made through the FUNCTIONAL API

In [ ]:
print(History.history.keys())

In [ ]:
plt.plot(History.history['loss'], label='Training Loss') 
plt.plot(History.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend() # Needs to be in a list
plt.show()

# Model Evaluation and Testing

In [ ]:
test_dataset

In [ ]:
test_dataset = test_dataset.batch(1) # Have to do this because we need to keep into account the dimension of the test dataset as well while evaluating.
lenet_model.evaluate(test_dataset)

# We have a 64.5% accuracy

In [ ]:
def parasite_or_not(x):
    if (x < 0.5):
        return 'P'
    else:
        return 'U'

In [ ]:
parasite_or_not(lenet_model.predict(test_dataset.take(1))[0][0]) 

'''# lenet_model.predict(test_dataset.take(1)) outputs:
[
  [0.85], # Image 0 prediction
  [0.12], # Image 1 prediction
  [0.44], # Image 2 prediction
  ...
  [0.91]  # Image 32 prediction
]
TensorFlow models output predictions in a 2D batch format [[0.85], [0.12]]. The first [0] selects the specific image in the batch, and the second [0] extracts the raw probability number 
from inside that vector.'''

In [ ]:
for i,(image,label) in enumerate(test_dataset.take(9)):

    ax = plt.subplot(3,3, i+1)
    plt.imshow(image[0])
    plt.title(str(parasite_or_not(label.numpy()[0])) + ':' + str(parasite_or_not(lenet_model.predict(image)[0][0])))
    plt.axis('off')

# Saving to and Loading from Google Drive

In [ ]:
from google.colab import drive # type: ignore
drive.mount('/content/drive')

In [ ]:
lenet_model.export('/content/drive/MyDrive/lenet_saved_directory')
drive_folder_path = '/content/drive/MyDrive/lenet_saved_directory'

# Load the directory as a specialized inference layer
lenet_loaded = tf.keras.layers.TFSMLayer(drive_folder_path, call_endpoint='serve')

print("Model loaded successfully from Google Drive!")

In [ ]:
lenet_model.save('/content/drive/MyDrive/lenet_malaria_saved_model.keras')
lenet_loaded = tf.keras.models.load_model('/content/drive/MyDrive/lenet_malaria_saved_model.keras')
lenet_loaded.summary()

In [ ]:
# Using the LOADED MODEL from google drive to do the predictions
for i,(image,label) in enumerate(test_dataset.take(9)):

    ax = plt.subplot(3,3, i+1)
    plt.imshow(image[0])
    plt.title(str(parasite_or_not(label.numpy()[0])) + ':' + str(parasite_or_not(lenet_loaded.predict(image)[0][0])))
    plt.axis('off')

In [ ]:
lenet_loaded.evaluate(test_dataset)

# Saving using the HDF5 method -> lightweight Model Saving Method

In [ ]:
# In a HDF5 format the configurations aren't stored, so you will have to compile the model again
lenet_model.save('/content/drive/MyDrive/lenet_malaria_model.h5')


In [ ]:
h5_model_path = '/content/drive/MyDrive/lenet_malaria_model.h5'
lenet_loaded_h5 = tf.keras.models.load_model(h5_model_path, compile=False) 
# If you only want to use the model for inference (making predictions) 
#and want to skip loading the optimizer, you can add compile=False to speed things up
# Setting compile=False tells Keras: "Just load the brain (weights and architecture). Do not waste time loading the training setup or rebuilding the optimizer."
lenet_loaded_h5.summary()

In [ ]:
# lenet_loaded_h5.evaluate(test_dataset) # In a HDF5 format the configurations aren't stored, so you will have to compile the model again

# ERROR GIVEN - You must call `compile()` before using the model.

#### Saving Just the Weights given that we already have the model's configurations

In [ ]:
# lenet_model.save_weights("weights/lenet_weights.weights.h5")

In [ ]:
# lenet_weights_model = lenet_model.load_weights("weights/lenet_weights")